In [3]:

from pathlib import Path
import hashlib
import re
import unicodedata, html, json
from collections import Counter
import pandas as pd
import numpy as np

In [4]:
PROJECT_ROOT = Path("../")
# Data directories
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
FILTERED_DIR = PROJECT_ROOT / "data" / "filtered"
FINAL_DIR = PROJECT_ROOT / "data" / "final"


for directory in [RAW_DIR, CLEANED_DIR, FILTERED_DIR, FINAL_DIR]:
    directory.mkdir(exist_ok=True, parents=True)

print(f"Raw directory is in {RAW_DIR.resolve()}")

Raw directory is in C:\LLM\education_llm\data\raw


In [6]:
!pip uninstall pyarrow datasets -y
!pip install --upgrade --force-reinstall pyarrow datasets


Found existing installation: pyarrow 25.0.1
Uninstalling pyarrow-25.0.1:
  Successfully uninstalled pyarrow-25.0.1
Found existing installation: datasets 5.0.1
Uninstalling datasets-5.0.1:
  Successfully uninstalled datasets-5.0.1
  Using cached pyarrow-25.0.1-cp313-cp313-win_amd64.whl.metadata (3.0 kB)
  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached filelock-3.32.6-py3-none-any.whl.metadata (2.0 kB)
  Using cached numpy-2.5.3-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-3.0.5-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached xxhash-4.0.1-cp313-cp313-win_amd64.whl.metadata (18 kB)
  Using cached multiprocess-0.70.19-py313-none-any.whl.metadata (7.5 kB)
  Using cached fsspec-2026.6.0

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from datasets import load_dataset

ds = load_dataset("CShorten/ML-ArXiv-Papers", split = "train")

print(ds)

c:\LLM\education_llm\venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jyusu\.cache\huggingface\hub\datasets--CShorten--ML-ArXiv-Papers. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 117592/117592 [00:01<00:00, 83069.39 examples/s]

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [8]:
ds

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})

In [21]:
len(ds['title'])

117592

In [22]:
len(ds['abstract'])

117592

In [24]:
raw_file = RAW_DIR/ 'arxivpaper.jsonl'

with open(raw_file, "w", encoding="utf-8") as f:
    for abstract in ds:
        record = {
            'abstract': abstract['abstract']
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
print(f"Saved raw data to: {raw_file}")
print(f"File size: {raw_file.stat().st_size / (1024**2):.2f} MB")

Saved raw data to: ..\data\raw\arxivpaper.jsonl
File size: 133.65 MB


In [29]:
def load_json(path):
    records = []

    with open(path, 'r', encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records

In [30]:
raw_docs = load_json(raw_file)

print("Number of raw documents:", len(raw_docs))

Number of raw documents: 117592


In [31]:
raw_char = sum(len(doc['abstract']) for doc in raw_docs)

print("Raw documents:", len(raw_docs))
print("Raw characters:", raw_char)
print("Raw size MB:", round(raw_file.stat().st_size / (1024**2), 2))

Raw documents: 117592
Raw characters: 136100194
Raw size MB: 133.65


## Cleaning 

In [32]:
def html_cleaning(abstract):

    abstract = re.sub(r"<[^>]+>", " ", abstract)
    return abstract

def normalize_unicode(abstract):
   
    return unicodedata.normalize("NFKC", abstract)

def remove_control_characters(abstract):
    
    return "".join(
        char for char in abstract
        if char in "\n\t" or not unicodedata.category(char).startswith("C")
    )
def normalize_whitespace(abstract):
    """
    Compress repeated whitespace.
    """
    abstract = re.sub(r"[ \t]+", " ", abstract)
    abstract = re.sub(r"\n\s*\n+", "\n\n", abstract)

    return abstract.strip()

def clean_text(abstract):
    """
    Complete cleaning pipeline.
    """
    text = html.unescape(abstract)
    text = html_cleaning(abstract)
    text = normalize_unicode(abstract)
    text = remove_control_characters(abstract)
    text = normalize_whitespace(abstract)

    return text

In [33]:
cleaned_docs = []

cleaning_stats = {
    "input": 0,
    "output": 0,
    "empty_removed": 0
}

for doc in raw_docs:

    cleaning_stats["input"] += 1

    original_text = doc["abstract"]
    cleaned_text = clean_text(original_text)

    if not cleaned_text:
        cleaning_stats["empty_removed"] += 1
        continue

    cleaned_docs.append({
        "abstract": cleaned_text
    })

cleaning_stats["output"] = len(cleaned_docs)

print(cleaning_stats)

{'input': 117592, 'output': 117592, 'empty_removed': 0}


In [35]:
cleaned_file = CLEANED_DIR/ 'cleaned_abstract.jsonl'

with open(cleaned_file, 'w', encoding='utf-8') as f:
    for doc in cleaned_docs:
        f.write(
            json.dumps(doc, ensure_ascii=False) + "\n"
        )
print(f"Saved cleaned data to: {cleaned_file}")

Saved cleaned data to: ..\data\cleaned\cleaned_abstract.jsonl


In [49]:
FILTER_CONFIG = {
    "min_chars": 100,
    "max_chars": 2000,
    "min_alpha_ratio": 0.30,
    "max_repeated_line_ratio": 0.30
}

FILTER_CONFIG

{'min_chars': 100,
 'max_chars': 2000,
 'min_alpha_ratio': 0.3,
 'max_repeated_line_ratio': 0.3}

In [50]:
def alphabetic_ratio(text):

    if not text:
        return 0.0

    alpha_count = sum(char.isalpha() for char in text)

    return alpha_count / len(text)


def repeated_line_ratio(text):
    
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if len(lines) <= 1:
        return 0.0

    counts = Counter(lines)

    repeated_lines = sum(
        count
        for count in counts.values()
        if count > 1
    )

    return repeated_lines / len(lines)

In [51]:
filtered_docs = []

filter_stats = {
    "input": len(cleaned_docs),
    "min_chars": 0,
    "max_chars": 0,
    "alpha_ratio": 0,
    "repeated_lines": 0,
    "output": 0
}

for doc in cleaned_docs:

    text = doc["abstract"]

    # Minimum length
    if len(text) < FILTER_CONFIG["min_chars"]:
        filter_stats["min_chars"] += 1
        continue

    # Maximum length
    if len(text) > FILTER_CONFIG["max_chars"]:
        filter_stats["max_chars"] += 1
        continue

    # Alphabetic ratio
    if alphabetic_ratio(text) < FILTER_CONFIG["min_alpha_ratio"]:
        filter_stats["alpha_ratio"] += 1
        continue

    # Repeated line ratio
    if repeated_line_ratio(text) > FILTER_CONFIG["max_repeated_line_ratio"]:
        filter_stats["repeated_lines"] += 1
        continue

    filtered_docs.append(doc)

filter_stats["output"] = len(filtered_docs)

print(filter_stats)

{'input': 117592, 'min_chars': 22, 'max_chars': 17, 'alpha_ratio': 0, 'repeated_lines': 1, 'output': 117552}


In [52]:
filtered_file = FILTERED_DIR / "arxivpapers_filtered.jsonl"

with open(filtered_file, "w", encoding="utf-8") as f:
    for doc in filtered_docs:
        f.write(
            json.dumps(doc, ensure_ascii=False) + "\n"
        )

print(f"Saved filtered data to: {filtered_file}")

Saved filtered data to: ..\data\filtered\arxivpapers_filtered.jsonl


In [53]:
def document_hash(text):
    
    normalized = text.strip().lower()

    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()

In [58]:
unique_docs = []
seen_hashes = set()
duplicated_lines = []

duplicate_count = 0

for doc in filtered_docs:

    text = doc["abstract"]

    doc_hash = document_hash(text)

    if doc_hash in seen_hashes:
        duplicate_count += 1
        duplicated_lines.append(text)
        continue

    seen_hashes.add(doc_hash)
    unique_docs.append(doc)

print("Input documents:", len(filtered_docs))
print("Duplicates removed:", duplicate_count)
print("Unique documents:", len(unique_docs))

Input documents: 117552
Duplicates removed: 4278
Unique documents: 113274


### Task, Watching whic lines are deupicated and not duplicated 10 of them 

In [59]:
duplicated_lines[:10]

['This paper uses Support Vector Machines (SVM) to fuse multiple classifiers\nfor an offline signature system. From the signature images, global and local\nfeatures are extracted and the signatures are verified with the help of\nGaussian empirical rule, Euclidean and Mahalanobis distance based classifiers.\nSVM is used to fuse matching scores of these matchers. Finally, recognition of\nquery signatures is done by comparing it with all signatures of the database.\nThe proposed system is tested on a signature database contains 5400 offline\nsignatures of 600 individuals and the results are found to be promising.',
 'The problem of clustering is considered, for the case when each data point is\na sample generated by a stationary ergodic process. We propose a very natural\nasymptotic notion of consistency, and show that simple consistent algorithms\nexist, under most general non-parametric assumptions. The notion of consistency\nis as follows: two samples should be put into the same cluste

In [60]:
unique_docs[:10]

[{'abstract': 'The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rate functions. The ideas are\nillustrated on the example of nonparametric regression in Gaussi

In [63]:
EMAIL_PATTERN = re.compile(
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)

PHONE_PATTERN = re.compile(
    r"(?<!\d)(?:\+?\d[\d\s().-]{7,}\d)(?!\d)"
)

NUMBER_PATTERN = re.compile(
    r"(?<!\d)\d{6,}(?!\d)"
)

In [64]:
def scrub_pii(text):
        stats = {
        "email": 0,
        "phone": 0,
        "digits": 0
    }

        text, stats["email"] = EMAIL_PATTERN.subn(
            "<EMAIL>",
            text
        )

        text, stats["phone"] = PHONE_PATTERN.subn(
            "<PHONE>",
            text
        )

        text, stats["digits"] = NUMBER_PATTERN.subn(
            "<NUMBER>",
            text
        )

        return text, stats

In [65]:
final_docs = []

pii_stats = {
    "email": 0,
    "phone": 0,
    "digits": 0
}

for doc in unique_docs:

    cleaned_text, stats = scrub_pii(
        doc["abstract"]
    )

    for key in pii_stats:
        pii_stats[key] += stats[key]

    final_docs.append({
        "abstract": cleaned_text
    })

print("PII matches:")
print(pii_stats)

PII matches:
{'email': 17, 'phone': 922, 'digits': 165}
